# Chapter 2: Forward pricing in code

Recomputes every worked example from Chapter 2 so you can change the inputs and see what happens.
Notes: [[chapters/02-forward-pricing]]

All formulas use **simple interest**, `(1 + r * t)`, as in the book. `t` is in years (months / 12) and `r` is a decimal (6% = 0.06).

In [1]:
def stock_forward(S, r, t, D=0.0):
    """Simplified stock forward: F = S(1 + rt) - D (all dividends lumped into D, no interest on them)."""
    return S * (1 + r * t) - D

def forward_with_cashflows(P, r, t, flows):
    """Exact forward for a stock or bond: F = P(1 + rt) - sum(cash_n * (1 + r_n * t_n)).
    flows is a list of (cash, rate, years_to_maturity_after_payment)."""
    return P * (1 + r * t) - sum(c * (1 + rn * tn) for c, rn, tn in flows)

def commodity_forward(C, r, t, s, i):
    """F = C(1 + rt) + s*t + i*t"""
    return C * (1 + r * t) + (s + i) * t

def commodity_cash_from_forward(F, r, t, s, i):
    """Solve the commodity formula for C."""
    return (F - (s + i) * t) / (1 + r * t)

def fx_forward(S, r_dom, r_for, t):
    """F = S (1 + r_d t) / (1 + r_f t), S = domestic units per 1 foreign unit."""
    return S * (1 + r_dom * t) / (1 + r_for * t)

def implied_spot(F, r, t, D):
    return (F + D) / (1 + r * t)

def implied_rate(F, S, t, D):
    return ((F + D) / S - 1) / t

def implied_dividend(F, S, r, t):
    return S * (1 + r * t) - F

## 1. Jerry's land (forward price = cash + costs - benefits)

In [2]:
cash, r = 100_000, 0.08
interest   = cash * r * 1
tax        = 2_000 + 2_000 * r * 3 / 12                      # loan for the last 3 months
oil        = 12 * 500
oil_interest = sum(500 * r * (12 - k) / 12 for k in range(1, 13))   # payment k earns for 12-k months
forward = cash + (interest + tax) - (oil + oil_interest)
print(f"costs of buying now    = {interest + tax:,.2f}")
print(f"benefits of buying now = {oil + oil_interest:,.2f}   (interest part: {oil_interest:.2f}; 1+...+11 = {sum(range(1, 12))})")
print(f"forward price          = {forward:,.2f}")
print(f"basis (cash - forward) = {cash - forward:,.2f}")

costs of buying now    = 10,040.00
benefits of buying now = 6,220.00   (interest part: 220.00; 1+...+11 = 66)
forward price          = 103,820.00
basis (cash - forward) = -3,820.00


## 2. Commodity, stock, bond and currency examples

In [3]:
# Commodity: solve for the cash price implied by a 3-month forward of 77.40
C = commodity_cash_from_forward(77.40, 0.08, 3/12, 3.00, 0.60)
print(f"commodity: implied cash price = {C:.2f}; convenience yield if cash is 76.25 = {76.25 - C:.2f}")
print(f"commodity: forward from that cash price = {commodity_forward(C, 0.08, 3/12, 3.00, 0.60):.2f}")

# Stock with two dividends (exact), then the simplified version
exact = forward_with_cashflows(67.00, 0.06, 8/12, [(0.33, 0.062, 7/12), (0.33, 0.065, 1/12)])
print(f"stock exact      F = {exact:.4f}")
print(f"stock simplified F = {stock_forward(67.00, 0.06, 8/12, D=0.66):.2f}")

# Bond
bond = forward_with_cashflows(109.76, 0.08, 10/12, [(5.25, 0.082, 8/12), (5.25, 0.085, 2/12)])
print(f"bond forward F = {bond:.4f}")

# Currency: 1 euro = 1.50 dollars, dollar rate 6%, euro rate 4%, six months
print(f"fx forward F = {fx_forward(1.50, 0.06, 0.04, 0.5):.4f}")

commodity: implied cash price = 75.00; convenience yield if cash is 76.25 = 1.25
commodity: forward from that cash price = 77.40
stock exact      F = 69.0063
stock simplified F = 69.02
bond forward F = 106.2160
fx forward F = 1.5147


## 3. Cash-and-carry arbitrage

In [4]:
S, r, t, D = 67.00, 0.06, 8/12, 0.66
fair = stock_forward(S, r, t, D)
market_F = 69.50
interest_cost = S * r * t
profit = -interest_cost - S + D + market_F      # book's list: interest, stock, dividend, forward price received
print(f"fair forward = {fair:.2f}, market forward = {market_F:.2f}")
print(f"interest cost on 67.00 for 8 months = {interest_cost:.2f}")
print(f"cash-and-carry profit = {profit:.2f}   (= market - fair = {market_F - fair:.2f})")

fair forward = 69.02, market forward = 69.50
interest cost on 67.00 for 8 months = 2.68
cash-and-carry profit = 0.48   (= market - fair = 0.48)


## 4. Implied values

In [5]:
F = 69.50
print(f"implied spot     = {implied_spot(F, r, t, D):.4f}")
print(f"implied rate     = {implied_rate(F, S, t, D):.4%}")
print(f"implied dividend = {implied_dividend(F, S, r, t):.2f}  (about {implied_dividend(F, S, r, t) / 2:.2f} per payment if two are paid)")

implied spot     = 67.4615
implied rate     = 7.0746%
implied dividend = 0.18  (about 0.09 per payment if two are paid)


## 5. Short sales: the no-arbitrage band

In [6]:
r_long, r_short = 0.06, 0.04
upper = stock_forward(S, r_long, t, D)
lower = stock_forward(S, r_short, t, D)
print(f"band for someone who must short stock: {lower:.2f} to {upper:.2f}")

market_F = 68.75
print(f"forward at {market_F}: stock owner gains {upper - market_F:+.2f}, non-owner (short rate) gains {lower - market_F:+.2f}")

band for someone who must short stock: 68.13 to 69.02
forward at 68.75: stock owner gains +0.27, non-owner (short rate) gains -0.62


## Try it
- Change the interest rate in section 3 and watch the fair forward and the arbitrage profit move.
- Raise `r_short` towards `r_long` and watch the band in section 5 narrow. At equal rates there is a single fair price.
- Set the oil revenue in section 1 to 2,000 a month and check the basis turns positive (+14,840).